# Tahap 1: Scraping Metadata Putusan MA

Notebook ini mengambil metadata putusan dari situs **putusan3.mahkamahagung.go.id** dan menyimpannya ke CSV.

- **Input:** Pilihan pengguna (kategori, sub-kategori, parameter scraping)
- **Output:** `../data/raw/metadata/metadata_[kategori]_[timestamp].csv`

> **Catatan:** Jika muncul error HTTP 403, naikkan nilai `DELAY_DETIK` atau coba di waktu berbeda.

In [9]:
%pip install requests beautifulsoup4 lxml pandas tqdm -q
print('Instalasi selesai!')

Note: you may need to restart the kernel to use updated packages.
Instalasi selesai!



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Konfigurasi

In [10]:
import requests
import pandas as pd
import time
import os
import re
from bs4 import BeautifulSoup
from tqdm.notebook import tqdm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

BASE_URL = 'https://putusan3.mahkamahagung.go.id'
OUTPUT_DIR = '../data/raw/metadata'
os.makedirs(OUTPUT_DIR, exist_ok=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'Accept-Language': 'id-ID,id;q=0.9,en-US;q=0.8',
    'Referer': 'https://putusan3.mahkamahagung.go.id/'
}

KATEGORI_PERDATA = {
    '1': {
        'nama': 'Perdata Umum',
        'deskripsi': 'Gugatan perdata umum: wanprestasi, PMH, waris, tanah, dll.',
        'url_path': '/direktori/index/kategori/perdata-1',
        'sub': {
            'A': {'nama': 'Semua Sub-Kategori', 'url_suffix': ''},
            'B': {'nama': 'Perbuatan Melawan Hukum', 'url_suffix': '/sub/perbuatan-melawan-hukum'},
            'C': {'nama': 'Wanprestasi', 'url_suffix': '/sub/wanprestasi'},
            'D': {'nama': 'Sengketa Tanah', 'url_suffix': '/sub/tanah'},
            'E': {'nama': 'Waris', 'url_suffix': '/sub/waris'},
            'F': {'nama': 'Perceraian (PN)', 'url_suffix': '/sub/perceraian'},
            'G': {'nama': 'Perjanjian', 'url_suffix': '/sub/perjanjian'},
        }
    },
    '2': {
        'nama': 'Perdata Khusus',
        'deskripsi': 'PHI, Kepailitan, HKI, Niaga, PKPU, dll.',
        'url_path': '/direktori/index/kategori/perdata-khusus',
        'sub': {
            'A': {'nama': 'Semua Sub-Kategori', 'url_suffix': ''},
            'B': {'nama': 'PHI (Hub. Industrial)', 'url_suffix': '', 'url_override': '/direktori/index/kategori/phi'},
            'C': {'nama': 'Kepailitan', 'url_suffix': '', 'url_override': '/direktori/index/kategori/kepailitan'},
            'D': {'nama': 'HKI', 'url_suffix': '', 'url_override': '/direktori/index/kategori/hki'},
            'E': {'nama': 'PKPU', 'url_suffix': '', 'url_override': '/direktori/index/kategori/pkpu'},
        }
    },
    '3': {
        'nama': 'Perdata Agama',
        'deskripsi': 'Perceraian, waris Islam, harta bersama, ekonomi syariah.',
        'url_path': '/direktori/index/kategori/perdata-agama-1',
        'sub': {
            'A': {'nama': 'Semua Sub-Kategori', 'url_suffix': ''},
            'B': {'nama': 'Perceraian', 'url_suffix': '', 'url_override': '/direktori/index/kategori/perceraian'},
            'C': {'nama': 'Waris Islam', 'url_suffix': '', 'url_override': '/direktori/index/kategori/waris-1'},
            'D': {'nama': 'Harta Bersama', 'url_suffix': '', 'url_override': '/direktori/index/kategori/harta-bersama'},
            'E': {'nama': 'Ekonomi Syariah', 'url_suffix': '', 'url_override': '/direktori/index/kategori/ekonomi-syariah'},
            'F': {'nama': 'Hadhanah', 'url_suffix': '', 'url_override': '/direktori/index/kategori/hadhanah'},
        }
    },
}

print('Konfigurasi dimuat.')
print('Base URL : ' + BASE_URL)
print('Output   : ' + os.path.abspath(OUTPUT_DIR))

Konfigurasi dimuat.
Base URL : https://putusan3.mahkamahagung.go.id
Output   : c:\cbrs\data\raw\metadata


## Input Pengguna: Pilih Kategori

In [ ]:
print('=' * 60)
print('  SCRAPING PUTUSAN MAHKAMAH AGUNG RI')
print('  Case-Based Reasoning (CBR)')
print('=' * 60)
print()
print('PILIH JENIS PERKARA PERDATA:')
print()
for kode, info in KATEGORI_PERDATA.items():
    print('  [' + kode + '] ' + info['nama'])
    print('       └─ ' + info['deskripsi'])
    print()
print('-' * 60)
PILIHAN_KATEGORI = input('Masukkan nomor pilihan (1/2/3): ').strip() or '1'
if PILIHAN_KATEGORI not in KATEGORI_PERDATA:
    print('Pilihan tidak valid, default ke Perdata Umum (1)')
    PILIHAN_KATEGORI = '1'
kategori = KATEGORI_PERDATA[PILIHAN_KATEGORI]
print('Kategori dipilih: [' + PILIHAN_KATEGORI + '] ' + kategori['nama'])

  SCRAPING PUTUSAN MAHKAMAH AGUNG RI
  Case-Based Reasoning (CBR)

PILIH JENIS PERKARA PERDATA:

  [1] Perdata Umum
       └─ Gugatan perdata umum: wanprestasi, PMH, waris, tanah, dll.

  [2] Perdata Khusus
       └─ PHI, Kepailitan, HKI, Niaga, PKPU, dll.

  [3] Perdata Agama
       └─ Perceraian, waris Islam, harta bersama, ekonomi syariah.

------------------------------------------------------------


## Input Pengguna: Pilih Sub-Kategori & Parameter

In [ ]:
print('=' * 60)
print('  SUB-KATEGORI: ' + kategori['nama'])
print('=' * 60)
for kode, sub_info in kategori['sub'].items():
    print('  [' + kode + '] ' + sub_info['nama'])
print()
PILIHAN_SUB = input('Kode sub-kategori: ').strip().upper() or 'A'
if PILIHAN_SUB not in kategori['sub']:
    print('Tidak valid, default ke Semua (A)')
    PILIHAN_SUB = 'A'
sub = kategori['sub'][PILIHAN_SUB]
print('Sub-kategori: ' + sub['nama'])

print()
print('PARAMETER SCRAPING (Enter = default):')
try:
    JUMLAH_HALAMAN = int(input('  Jumlah halaman [default=3, max=50]: ').strip() or '3')
    JUMLAH_HALAMAN = min(max(1, JUMLAH_HALAMAN), 50)
except ValueError:
    JUMLAH_HALAMAN = 3

try:
    DELAY_DETIK = float(input('  Delay antar request detik [default=2]: ').strip() or '2')
    DELAY_DETIK = max(1.0, DELAY_DETIK)
except ValueError:
    DELAY_DETIK = 2.0

TAHUN_FILTER = input('  Filter tahun [kosong=semua, contoh: 2023]: ').strip()
KEYWORD_FILTER = input('  Filter kata kunci judul [kosong=semua]: ').strip()

if 'url_override' in sub and sub['url_override']:
    BASE_SCRAPE_URL = BASE_URL + sub['url_override']
else:
    BASE_SCRAPE_URL = BASE_URL + kategori['url_path'] + sub.get('url_suffix', '')
if TAHUN_FILTER:
    BASE_SCRAPE_URL += '/tahunjenis/putus/tahun/' + TAHUN_FILTER

print()
print('RINGKASAN:')
print('  Kategori    : ' + kategori['nama'] + ' - ' + sub['nama'])
print('  Halaman     : ' + str(JUMLAH_HALAMAN))
print('  Delay       : ' + str(DELAY_DETIK) + ' detik')
print('  Tahun       : ' + (TAHUN_FILTER or 'semua'))
print('  Keyword     : ' + (KEYWORD_FILTER or 'semua'))
print('  URL         : ' + BASE_SCRAPE_URL)

  SUB-KATEGORI: Perdata Umum
  [A] Semua Sub-Kategori
  [B] Perbuatan Melawan Hukum
  [C] Wanprestasi
  [D] Sengketa Tanah
  [E] Waris
  [F] Perceraian (PN)
  [G] Perjanjian

Sub-kategori: Semua Sub-Kategori

PARAMETER SCRAPING (Enter = default):

RINGKASAN:
  Kategori    : Perdata Umum - Semua Sub-Kategori
  Halaman     : 50
  Delay       : 30.0 detik
  Tahun       : semua
  Keyword     : semua
  URL         : https://putusan3.mahkamahagung.go.id/direktori/index/kategori/perdata-1


## Fungsi Scraping

In [ ]:
def ambil_halaman(url, retries=3):
    for i in range(retries):
        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 200:
                return BeautifulSoup(resp.content, 'lxml')
            print('    Status ' + str(resp.status_code) + ' pada ' + url)
        except requests.RequestException as e:
            print('    Error (percobaan ' + str(i+1) + '/' + str(retries) + '): ' + str(e))
            time.sleep(DELAY_DETIK * 2)
    return None


def ekstrak_nomor(teks):
    m = re.search(r'Nomor\s+([\w\s/.-]+?)(?:\s*·|\s*Tanggal|\s*—|$)', teks, re.IGNORECASE)
    if m:
        return m.group(1).strip()[:80]
    m2 = re.search(r'(\d+\s*/\s*(?:PDT|Pdt|AG|PID)[\w./\s-]*)', teks)
    return m2.group(1).strip() if m2 else ''


def ekstrak_tanggal(teks):
    m = re.search(r'(?:Putus|Tanggal)\s*[:\-\u2013]?\s*(\d{1,2}\s+\w+\s+\d{4})', teks, re.IGNORECASE)
    if m:
        return m.group(1).strip()
    m2 = re.search(r'(\d{1,2}-\d{2}-\d{4})', teks)
    return m2.group(1) if m2 else ''


def ekstrak_pengadilan(teks):
    m = re.search(r'Pengadilan\s+([A-Z\s]+?)(?:\s+Perdata|\s+Pidana|\s*·|\s*$)', teks)
    if m:
        return m.group(1).strip()[:60]
    if 'MAHKAMAH AGUNG' in teks.upper():
        return 'MAHKAMAH AGUNG'
    return ''


def ekstrak_para_pihak(teks):
    m = re.search(
        r'([A-Z][\w\s,.]+?)\s+(?:VS|vs|melawan|MELAWAN)\s+([A-Z][\w\s,.]+?)(?:\s*·|\s*\d+\s*\u2014|$)',
        teks
    )
    if m:
        return m.group(1).strip()[:100] + ' VS ' + m.group(2).strip()[:100]
    return ''


def parse_putusan_dari_halaman(soup):
    putusan_list = []
    for link in soup.find_all('a', href=True):
        href = link['href']
        if '/direktori/putusan/' not in href:
            continue
        teks_link = link.get_text(strip=True)
        if not teks_link:
            continue
        try:
            konteks = link.parent.parent.get_text(separator=' | ', strip=True)[:500]
        except Exception:
            konteks = teks_link
        full_url = BASE_URL + href if href.startswith('/') else href
        entry = {
            'judul'      : teks_link,
            'url_detail' : full_url,
            'nomor'      : ekstrak_nomor(konteks),
            'tanggal'    : ekstrak_tanggal(konteks),
            'pengadilan' : ekstrak_pengadilan(konteks),
            'para_pihak' : ekstrak_para_pihak(konteks),
            'klasifikasi': sub['nama'],
            'kategori'   : kategori['nama'],
        }
        if KEYWORD_FILTER:
            gabung = (entry['judul'] + ' ' + entry['para_pihak']).lower()
            if KEYWORD_FILTER.lower() not in gabung:
                continue
        putusan_list.append(entry)
    seen, unik = set(), []
    for p in putusan_list:
        if p['url_detail'] not in seen and p['url_detail']:
            seen.add(p['url_detail'])
            unik.append(p)
    return unik


print('Fungsi scraping siap.')

Fungsi scraping siap.


## Jalankan Scraping

In [ ]:
print('=' * 60)
print('  MULAI SCRAPING METADATA')
print('=' * 60)

semua_putusan = []
gagal_halaman = []

for no_hal in tqdm(range(1, JUMLAH_HALAMAN + 1), desc='Scraping halaman'):
    if no_hal == 1:
        url_hal = BASE_SCRAPE_URL + '.html'
    else:
        url_hal = BASE_SCRAPE_URL + '/page/' + str(no_hal) + '.html'

    print('\n  Halaman ' + str(no_hal) + ': ' + url_hal)
    soup = ambil_halaman(url_hal)

    if not soup:
        print('  Gagal halaman ' + str(no_hal))
        gagal_halaman.append(no_hal)
        continue

    hasil = parse_putusan_dari_halaman(soup)
    print('  Ditemukan ' + str(len(hasil)) + ' putusan')

    if not hasil:
        print('  Tidak ada data, kemungkinan halaman terakhir.')
        break

    semua_putusan.extend(hasil)
    time.sleep(DELAY_DETIK)

print()
print('=' * 60)
print('  TOTAL: ' + str(len(semua_putusan)) + ' putusan berhasil dikumpulkan')
if gagal_halaman:
    print('  Halaman gagal: ' + str(gagal_halaman))
print('=' * 60)

  MULAI SCRAPING METADATA


ImportError: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html

## Simpan Metadata ke CSV

In [ ]:
if not semua_putusan:
    print('Tidak ada data untuk disimpan.')
else:
    df = pd.DataFrame(semua_putusan)
    df['id'] = range(1, len(df) + 1)
    df['timestamp_scrape'] = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    def nama_aman(s, n=40):
        return re.sub(r'[\s<>:"/\\|?*]+', '_', str(s))[:n]

    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    nama_file = 'metadata_' + nama_aman(kategori['nama']) + '_' + nama_aman(sub['nama']) + '_' + ts + '.csv'
    path_csv = os.path.join(OUTPUT_DIR, nama_file)
    df.to_csv(path_csv, index=False, encoding='utf-8-sig')

    print('Metadata disimpan ke: ' + path_csv)
    print('Total baris         : ' + str(len(df)))
    print()
    print('Preview 5 data pertama:')
    display(df.head())